# FINN - End-to-End Flow (CNN CIFAR-10)
-----------------------------------------------------------------

In this notebook, we replace the original **TFC-w1a1 model (MNIST)** with a **binarized CNN trained on CIFAR-10** using Brevitas. The FINN pipeline (Network preparation, Hardware build, PYNQ deployment) is kept **identical** to the original notebook.

| | Originale | Questo notebook |
|---|---|---|
| Modello | TFC-w1a1 (Fully Connected) | CNN-w1a1 (Convoluzionale) |
| Dataset | MNIST 28×28×1 | CIFAR-10 32×32×3 |
| Framework training | Brevitas (pretrained) | Brevitas (training da zero) |
| Pipeline FINN | invariata | invariata |


## Overview

The FINN compiler comes with many *transformations* that modify the ONNX representation of the network according to certain patterns. This notebook will demonstrate a *possible* sequence of such transformations to take a particular trained network all the way down to hardware, as shown in the figure below.

![](finn-design-flow-example.svg)

The white fields show the state of the network representation in the respective step. The colored fields represent the transformations that are applied to the network to achieve a certain result. The diagram is divided into 5 sections represented by a different color, each of it includes several flow steps. The flow starts in top left corner with Brevitas export (green section), followed by the preparation of the network (blue section) to bring the network into a form in which each layer can be represented by either a Vitis HLS function or a Verilog module. The model then gets passed to Vivado IPI stitching (orange section), and finally a PYNQ overlay bitfile is built and can be tested on a PYNQ board (yellow section).
There is an additional section for functional verification (red section) on the right side of the diagram, which we will not cover in this notebook. For details please take a look in the verification notebook which you can find [here](tfc_end2end_verification.ipynb)


This Jupyter notebook is organized based on the sections described above. We will use the following helper functions, `showSrc` to show source code of FINN library calls and `showInNetron` to show the ONNX model at the current transformation step. The Netron displays are interactive, but they only work when running the notebook actively and not on GitHub (i.e. if you are viewing this on GitHub you'll only see blank squares).

In [ ]:
from finn.util.visualization import showSrc, showInNetron
from finn.util.basic import make_build_dir
import os
    
build_dir = os.environ["FINN_BUILD_DIR"]

## Outline
-------------
1. [CNN CIFAR-10 — Training e Brevitas export](#brev_exp)
2. [Network preparation](#nw_prep)
3. [Hardware build](#vivado)
4. [PYNQ deployment](#hw_test)

## 1. CNN CIFAR-10 — Training e Brevitas export <a id='brev_exp'></a>

Instead of using the pretrained TFC-w1a1 model on MNIST, we define and train a **binarized CNN (W1A1) on CIFAR-10** using Brevitas.

### Architettura CNN-w1a1 per CIFAR-10
```
Input (1, 3, 32, 32)  — CIFAR-10: 3 canali colore, 32×32 px
  ├── QuantHardTanh (input quantization UINT8 → bipolar)
  ├── QuantConv2d(3→64,  3×3, pad=1) → BatchNorm → QuantHardTanh  W1A1
  ├── MaxPool2d(2×2)                                               → 16×16×64
  ├── QuantConv2d(64→128, 3×3, pad=1) → BatchNorm → QuantHardTanh W1A1
  ├── MaxPool2d(2×2)                                               →  8×8×128
  ├── QuantConv2d(128→256, 3×3, pad=1) → BatchNorm → QuantHardTanh W1A1
  ├── MaxPool2d(2×2)                                               →  4×4×256
  ├── Flatten                                                      →  4096
  ├── QuantLinear(4096→512) → BatchNorm → QuantHardTanh            W1A1
  └── QuantLinear(512→10)  → output logits
```

> **Nota**: lo schema `QuantConv2d → BatchNorm → QuantHardTanh` è lo stesso pattern W1A1 usato dal TFC originale (`QuantLinear → BatchNorm → QuantHardTanh`), compatibile con il flusso di streamlining FINN.

### 1.1 Installazione dipendenze aggiuntive

In [ ]:
import subprocess, sys

# brevitas è già presente nel container FINN, verifichiamo
try:
    import brevitas
    print(f"Brevitas già disponibile: {brevitas.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "brevitas", "-q"])
    print("Brevitas installata")

# torchvision per il dataset CIFAR-10
try:
    import torchvision
    print(f"Torchvision già disponibile: {torchvision.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torchvision", "-q"])
    print("Torchvision installata")

### 1.2 Dataset CIFAR-10

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing: normalizzazione standard CIFAR-10
# ToTensor() divide per 255 → [0,1], poi normalize → [-1,1]
# Questo è lo stesso approccio del TFC originale (ToTensor → divide by 255)
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),                            # [0,1]
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5)),            # [-1,1]
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5)),
])

DATA_DIR = "/tmp/cifar10_data"
os.makedirs(DATA_DIR, exist_ok=True)

trainset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False, num_workers=2)

CLASS_NAMES = ['aereo','automobile','uccello','gatto','cervo',
               'cane','rana','cavallo','nave','camion']

print(f"Train samples: {len(trainset)} | Test samples: {len(testset)}")

# Visualizza alcuni campioni
imgs, labels = next(iter(trainloader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    img = imgs[i].numpy().transpose(1,2,0)
    img = (img * 0.5 + 0.5).clip(0,1)   # de-normalize
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Esempi CIFAR-10')
plt.tight_layout()
plt.show()

### 1.3 Definizione della CNN binarizzata (W1A1) con Brevitas

Usiamo gli stessi building block del TFC originale:
- `QuantConv2d` con `weight_bit_width=1` (pesi binari)
- `QuantHardTanh` con `act_bit_width=1` (attivazioni binarie)
- `BatchNorm2d` tra Conv e attivazione (necessario per streamlining FINN)

In [ ]:
import brevitas
import brevitas.nn as qnn
from brevitas.nn import QuantConv2d, QuantLinear, QuantIdentity
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat

# Brevitas 0.10.0 — nomi esatti verificati
print(f'Brevitas version: {brevitas.__version__}')

def make_quant_conv(in_ch, out_ch, kernel, padding=1):
    """QuantConv2d W1 — pesi binari."""
    return QuantConv2d(
        in_ch, out_ch, kernel,
        padding=padding,
        bias=False,
        weight_bit_width=1,
        weight_quant=Int8WeightPerTensorFloat,
        return_quant_tensor=True
    )

def make_quant_linear(in_f, out_f, return_quant=True):
    """QuantLinear W1 — pesi binari."""
    return QuantLinear(
        in_f, out_f,
        bias=False,
        weight_bit_width=1,
        weight_quant=Int8WeightPerTensorFloat,
        return_quant_tensor=return_quant
    )

def make_quant_act():
    """Attivazione quantizzata A1 — QuantIdentity con bit_width=1."""
    return QuantIdentity(
        act_quant=Int8ActPerTensorFloat,
        bit_width=1,
        return_quant_tensor=True
    )


class CNNw1a1_CIFAR10(nn.Module):
    """
    CNN binarizzata W1A1 per CIFAR-10, compatibile con il flusso FINN.
    Schema: QuantConv2d(W1) → BatchNorm → QuantReLU(A1) → MaxPool
    Identico al pattern del TFC originale: QuantLinear → BN → QuantHardTanh.
    """
    def __init__(self, num_classes=10):
        super().__init__()

        # Quantizzazione input: porta i valori float nel range quantizzato
        self.input_quant = QuantIdentity(
            bit_width=8,
            return_quant_tensor=True
        )

        # === Blocco 1: Conv 3→64, 3×3 ===
        self.conv1   = make_quant_conv(3,   64, 3)
        self.bn1     = nn.BatchNorm2d(64)
        self.act1    = make_quant_act()
        self.pool1   = nn.MaxPool2d(2, 2)           # 16×16×64

        # === Blocco 2: Conv 64→128, 3×3 ===
        self.conv2   = make_quant_conv(64,  128, 3)
        self.bn2     = nn.BatchNorm2d(128)
        self.act2    = make_quant_act()
        self.pool2   = nn.MaxPool2d(2, 2)           # 8×8×128

        # === Blocco 3: Conv 128→256, 3×3 ===
        self.conv3   = make_quant_conv(128, 256, 3)
        self.bn3     = nn.BatchNorm2d(256)
        self.act3    = make_quant_act()
        self.pool3   = nn.MaxPool2d(2, 2)           # 4×4×256

        # === Classificatore FC ===
        self.fc1     = make_quant_linear(4*4*256, 512)
        self.bn_fc1  = nn.BatchNorm1d(512)
        self.act_fc1 = make_quant_act()

        # Output: pesi binari, nessuna quantizzazione sull'uscita
        self.fc2     = make_quant_linear(512, num_classes, return_quant=False)

    def _unpack(self, x):
        """Estrae il tensore float da un QuantTensor se necessario."""
        return x.value if hasattr(x, 'value') else x

    def forward(self, x):
        x = self.input_quant(x)

        # Blocco 1
        x = self.conv1(x)
        x = self.bn1(self._unpack(x))
        x = self.act1(x)
        x = self.pool1(self._unpack(x))

        # Blocco 2
        x = self.conv2(x)
        x = self.bn2(self._unpack(x))
        x = self.act2(x)
        x = self.pool2(self._unpack(x))

        # Blocco 3
        x = self.conv3(x)
        x = self.bn3(self._unpack(x))
        x = self.act3(x)
        x = self.pool3(self._unpack(x))

        # Flatten + FC
        x = self._unpack(x).view(-1, 4*4*256)
        x = self.fc1(x)
        x = self.bn_fc1(self._unpack(x))
        x = self.act_fc1(x)
        x = self.fc2(self._unpack(x))
        return x


cnn = CNNw1a1_CIFAR10()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cnn = cnn.to(device)

total_params = sum(p.numel() for p in cnn.parameters())
print(f'Dispositivo: {device}')
print(f'Parametri totali: {total_params:,}')
print(cnn)

### 1.4 Training della CNN W1A1 su CIFAR-10

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
CHECKPOINT_PATH = build_dir + "/cnn_cifar10_w1a1.pth"

train_acc_history = []
test_acc_history  = []

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

best_acc = 0.0

for epoch in range(EPOCHS):
    cnn.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    scheduler.step()

    train_acc = 100. * correct / total
    test_acc  = evaluate(cnn, testloader)
    train_acc_history.append(train_acc)
    test_acc_history.append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(cnn.state_dict(), CHECKPOINT_PATH)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  "
              f"Loss: {running_loss/len(trainloader):.3f}  "
              f"Train acc: {train_acc:.1f}%  "
              f"Test acc: {test_acc:.1f}%  "
              f"Best: {best_acc:.1f}%")

print(f"\nTraining completato. Miglior test accuracy: {best_acc:.2f}%")

In [ ]:
# Curve di training
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_acc_history, label='Train accuracy')
ax.plot(test_acc_history,  label='Test accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('CNN W1A1 — CIFAR-10 Training')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

# Carica il best checkpoint
cnn.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
final_acc = evaluate(cnn, testloader)
print(f"Accuratezza finale (best checkpoint): {final_acc:.2f}%")

### 1.5 Brevitas export in QONNX

Questo step è **identico** all'originale — sostituiamo solo il modello passato a `export_qonnx`. Il resto del flusso FINN non cambia.

In [ ]:
import onnx
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup

# Path ONNX — rinominiamo per chiarezza (il resto del notebook usa questa variabile)
export_onnx_path = build_dir + "/cnn_cifar10_w1_a1.onnx"

# Input dummy: batch=1, canali=3 (RGB), 32×32 — CIFAR-10
dummy_input = torch.randn(1, 3, 32, 32).to(device)

cnn.eval()
export_qonnx(cnn, dummy_input, export_onnx_path)  # semicolon per sopprimere log
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)

print(f"Modello esportato in QONNX: {export_onnx_path}")
print(f"Dimensione file: {os.path.getsize(export_onnx_path)/1024:.1f} KB")

In [ ]:
showInNetron(export_onnx_path)

In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())

In [ ]:
model.save(build_dir + "/cnn_cifar10_w1_a1_finn.onnx")
showInNetron(build_dir + "/cnn_cifar10_w1_a1_finn.onnx")

## 2. Network preparation <a id='nw_prep'></a>

* [FINN-style Dataflow Architectures](#dataflow_arch)
* [Tidy-up transformations](#basic_trafo)
* [Streamlining](#streamline)
* [Conversion to HW layers](#hw_layers)
* [Creating a Dataflow Partition](#dataflow_partition)
* [Specialize layers](#specialize_layers)
* [Folding and Datawidth Converter, FIFO and TLastMarker Insertion](#folding)


In this section, we will put the network through a series of transformations that puts it in a form that can be stitched together to form a FINN-style dataflow architecture, yielding a high-performance, high-efficiency FPGA accelerator.

### FINN-style Dataflow Architectures <a id='dataflow_arch'></a>

We start with a quick recap of FINN-style dataflow architectures. The key idea in such architectures is to parallelize across layers as well as within layers by dedicating a proportionate amount of compute resources to each layer, as illustrated in the figure below taken from the [FINN-R paper](https://arxiv.org/pdf/1809.04570.pdf):

![](finn-hw-arch.png)

In practice, the compute arrays are instantiated by function calls to optimized Vitis HLS building blocks from the [finn-hlslib](https://github.com/Xilinx/finn-hlslib) library or by Verilog modules from the [finn-rtllib](https://github.com/Xilinx/finn/tree/main/finn-rtllib). As these function calls/modules can only handle certain patterns/cases, we need to transform the network into an appropriate form so that we can replace network layers with these function calls/modules, which is the goal of the network preparation process.

### Tidy-up transformations <a id='basic_trafo'></a>
This section deals with some basic transformations, which are applied to the model like a kind of "tidy-up" to make it easier to be processed. They do not appear in the diagram above, but they are applied in many steps in the FINN flow to postprocess the model after a transformation and/or to prepare it for the next transformation.

These transformations are:
* GiveUniqueNodeNames
* GiveReadableTensorNames
* InferShapes
* InferDataTypes
* FoldConstants
* RemoveStaticGraphInputs

In the first two transformations (`GiveUniqueNodeNames`, `GiveReadableTensorNames`) the nodes in the graph are first given unique (by enumeration) names, then the tensors are given human-readable names (based on the node names). The following two transformations (`InferShapes`, `InferDataTypes`) derive the shapes and data types of the tensors from the model properties and set them in the `ValueInfo` of the model. These transformations can almost always be applied without negative effects and do not affect the structure of the graph, ensuring that all the information needed is available.

The next listed transformation is `FoldConstants`, which performs constant folding. It identifies a node with constant inputs and determines its output. The result is then set as constant-only inputs for the following node and the old node is removed. Although this transformation changes the structure of the model, it is a transformation that is usually always desired and can be applied to any model. And finally, we have `RemoveStaticGraphInputs` to remove any top-level graph inputs that already have ONNX initializers associated with them. 

These transformations can be imported and applied as follows.

In [ ]:
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

model.save(build_dir+"/cnn_cifar10_w1_a1_tidy.onnx")

In [ ]:
showInNetron(build_dir+"/cnn_cifar10_w1_a1_tidy.onnx")

### Adding Pre- and Postprocessing <a id='prepost'></a>

In many cases, it's common to apply some preprocessing to the raw data in a machine learning framework prior to training. For image classification networks, this may include conversion of raw 8-bit RGB values into floating point values between 0 and 1. Similarly, at the output of the network some postprocessing may be performed during deployment, such as extracting the indices of the classifications with the largest value (top-K indices).

In FINN, we can bake some of these pre/postprocessing operatings into the graph, and in some cases these can be highly beneficial for performance by allowing our accelerator to directly consume raw data instead of going through CPU preprocessing. 

We'll demonstrate this for our CNN CIFAR-10 network. Brevitas preprocesses BNN-PYNQ network inputs with `torchvision.transforms.ToTensor()` prior to training, which converts 8-bit RGB values into floats between 0 and 1 by dividing the input by 255. We can achieve the same effect in FINN by exporting a single-node ONNX graph for division by 255 (which already exists as `finn.util.pytorch.ToTensor`) and merging this with our original model. Finally, we're going to mark our input tensor as 8-bit to let FINN know which level of precision to use.

In [ ]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType

model = ModelWrapper(build_dir+"/cnn_cifar10_w1_a1_tidy.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name = build_dir+"/cnn_cifar10_w1_a1_preproc.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

model.save(build_dir+"/cnn_cifar10_w1_a1_with_preproc.onnx")
showInNetron(build_dir+"/cnn_cifar10_w1_a1_with_preproc.onnx")

In [ ]:
from qonnx.transformation.insert_topk import InsertTopK

# postprocessing: insert Top-1 node at the end
model = model.transform(InsertTopK(k=1))
chkpt_name = build_dir+"/cnn_cifar10_w1_a1_pre_post.onnx"
# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

showInNetron(build_dir+"/cnn_cifar10_w1_a1_pre_post.onnx")

### Streamlining <a id='streamline'></a>

Streamlining is a transformation containing several sub-transformations. The goal of streamlining is to eliminate floating point operations by moving them around, then collapsing them into one operation and in the last step transform them into multi-thresholding nodes. For more information on the theoretical background of this, see [this paper](https://arxiv.org/pdf/1709.04060).

Let's have a look at which sub-transformations `Streamline` consists of:

In [ ]:
from finn.transformation.streamline import Streamline
showSrc(Streamline)

In [ ]:
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(build_dir+"/cnn_cifar10_w1_a1_pre_post.onnx")
# move initial Mul (from preproc) past the Reshape
model = model.transform(MoveScalarLinearPastInvariants())
# streamline
model = model.transform(Streamline())
model.save(build_dir+"/cnn_cifar10_w1_a1_streamlined.onnx")
showInNetron(build_dir+"/cnn_cifar10_w1_a1_streamlined.onnx")

In [ ]:
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(absorb.AbsorbAddIntoMultiThreshold())
model = model.transform(absorb.AbsorbMulIntoMultiThreshold())
# absorb final add-mul nodes into TopK
model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(RoundAndClipThresholds())

# bit of tidy-up
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())

model.save(build_dir+"/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx")
showInNetron(build_dir+"/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx")

### Conversion to HW layers <a id='hw_layers'></a>
Converts the nodes to HW layers, these layers are abstraction layers that do not directly correspond to an HLS or Verilog implementation but they will be converted in either one later in the flow. In our case this transformation converts pairs of binary XnorPopcountMatMul layers to MVAU layers (matrix vector activation unit). Any immediately following MultiThreshold layers will also be absorbed into the MVAU.

Below is the code for the transformation and the network is visualized using netron to create the new structure with `MVAU` nodes.

In [ ]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
model = ModelWrapper(build_dir+"/cnn_cifar10_w1a1_ready_for_hw_conversion.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
# TopK to LabelSelect
model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model.save(build_dir+"/cnn_cifar10_w1_a1_hw_layers.onnx")
showInNetron(build_dir+"/cnn_cifar10_w1_a1_hw_layers.onnx")

### Creating a Dataflow Partition <a id='dataflow_partition'></a>

In the graph above, you can see that there is a mixture of FINN HW layers (`MVAU` and `Thresholding`) with one regular ONNX layers (Reshape). To create a bitstream, FINN needs a model with only HW layers. In order to achieve this, we will use the `CreateDataflowPartition` transformation to create a "dataflow partition" in this graph, separating out the HLS layers into another model, and replacing them with a placeholder layer called StreamingDataflowPartition.

In [ ]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition

model = ModelWrapper(build_dir+"/cnn_cifar10_w1_a1_hw_layers.onnx")
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir+"/cnn_cifar10_w1_a1_dataflow_parent.onnx")
showInNetron(build_dir+"/cnn_cifar10_w1_a1_dataflow_parent.onnx")

In [ ]:
from qonnx.custom_op.registry import getCustomOp
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
showInNetron(dataflow_model_filename)

In [ ]:
model = ModelWrapper(dataflow_model_filename)

### Specialize layers <a id='specialize_layers'></a>

In [ ]:
thresh_node = model.get_nodes_by_op_type("Thresholding")[0]
thresh_node_inst = getCustomOp(thresh_node)
thresh_node_inst.set_nodeattr("preferred_impl_style", "hls")

In [ ]:
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers

model = model.transform(SpecializeLayers("xc7z020clg400-1"))
model.save(build_dir + "/cnn_cifar10_w1_a1_specialize_layers.onnx")
showInNetron(build_dir + "/cnn_cifar10_w1_a1_specialize_layers.onnx")

### Folding <a id='folding'></a>

In [ ]:
model = ModelWrapper(build_dir + "/cnn_cifar10_w1_a1_specialize_layers.onnx")
fc_layers = model.get_nodes_by_op_type("MVAU_hls")
# each tuple is (PE, SIMD, ram_style) for each layer
# adjust based on resource budget of target FPGA
folding = [
    (16, 3,  "block"),   # conv1 input: SIMD=3 (RGB channels)
    (16, 64, "block"),   # conv2
    (16, 64, "block"),   # conv3
    (8,  64, "block"),   # fc1
    (10, 8,  "auto"),    # fc2 (output)
]
for fcl, (pe, simd, ramstyle) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.set_nodeattr("SIMD", simd)
    fcl_inst.set_nodeattr("ram_style", ramstyle)

# set input thresholding folding
thresh_layers = model.get_nodes_by_op_type("Thresholding_hls")
if thresh_layers:
    getCustomOp(thresh_layers[0]).set_nodeattr("PE", 3)   # 3 canali RGB

model.save(build_dir + "/cnn_cifar10_w1_a1_folded.onnx")
showInNetron(build_dir + "/cnn_cifar10_w1_a1_folded.onnx")

In [ ]:
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from finn.transformation.fpgadataflow.insert_fifo import InsertFIFO
from finn.transformation.fpgadataflow.insert_tlastmarker import InsertTLastMarker
from finn.transformation.fpgadataflow.prepare_ip import PrepareIP
from finn.transformation.fpgadataflow.hlssynth_ip import HLSSynthIP
from finn.transformation.fpgadataflow.set_exec_mode import SetExecMode

model = ModelWrapper(build_dir + "/cnn_cifar10_w1_a1_folded.onnx")
model = model.transform(InsertDWC())
model = model.transform(InsertFIFO())
model = model.transform(InsertTLastMarker())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(PrepareIP("xc7z020clg400-1", 10))
model = model.transform(HLSSynthIP())
model = model.transform(SetExecMode("rtlsim"))
model.save(build_dir + "/cnn_cifar10_w1_a1_ipgen.onnx")
showInNetron(build_dir + "/cnn_cifar10_w1_a1_ipgen.onnx")

## 3. Hardware build <a id='vivado'></a>

In [ ]:
from finn.transformation.fpgadataflow.create_stitched_ip import CreateStitchedIP

model = ModelWrapper(build_dir + "/cnn_cifar10_w1_a1_ipgen.onnx")
model = model.transform(CreateStitchedIP("xc7z020clg400-1", 10))
model.save(build_dir + "/cnn_cifar10_w1_a1_stitched_ip.onnx")
print("Stitched IP path: " + model.get_metadata_prop("vivado_stitch_proj"))

In [ ]:
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver

model = model.transform(MakePYNQDriver("zynq-iodma"))
model.save(build_dir + "/cnn_cifar10_w1_a1_pynq_driver.onnx")

In [ ]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

model = ModelWrapper(build_dir + "/cnn_cifar10_w1_a1_pynq_driver.onnx")
model = model.transform(ZynqBuild(platform="Pynq-Z1", period_ns=10))
model.save(build_dir + "/cnn_cifar10_w1_a1_post_synthesis.onnx")
print("Bitfile: " + model.get_metadata_prop("bitfile"))

## 4. PYNQ deployment <a id='hw_test'></a>

In [ ]:
from shutil import copy, copytree

# create directory for deployment files
deployment_dir = make_build_dir(prefix="pynq_deployment_")

# get and copy necessary files
# .bit and .hwh file
bitfile  = model.get_metadata_prop("bitfile")
hwh_file = model.get_metadata_prop("hw_handoff")
deploy_files = [bitfile, hwh_file]

for dfile in deploy_files:
    if dfile is not None:
        copy(dfile, deployment_dir)

# driver.py and python libraries
pynq_driver_dir = model.get_metadata_prop("pynq_driver_dir")
copytree(pynq_driver_dir, deployment_dir, dirs_exist_ok=True)

Next to these files, we will also need an example numpy array to test the network on the PYNQ board. We take a CIFAR-10 test image and save it as .npy file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Prendi un'immagine CIFAR-10 dal testset (raw uint8, non normalizzata)
raw_testset = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=False,
                                            transform=transforms.ToTensor())
x_sample, y_sample = raw_testset[0]
# Converte in uint8 [0,255] — l'FPGA riceve input grezzi UINT8
x_np = (x_sample.numpy() * 255).astype(np.uint8)    # (3, 32, 32)
x_np = np.expand_dims(x_np, axis=0)                 # (1, 3, 32, 32)

plt.imshow(x_np[0].transpose(1,2,0))
plt.title(f"Classe: {CLASS_NAMES[y_sample]}")
plt.axis('off')
plt.show()

model = ModelWrapper(build_dir + "/cnn_cifar10_w1_a1_post_synthesis.onnx")
iname  = model.graph.input[0].name
ishape = model.get_tensor_shape(iname)
print("Expected network input shape is " + str(ishape))
np.save(deployment_dir + "/input.npy", x_np.reshape(ishape))

In [ ]:
! ls {deployment_dir}

In [ ]:
from shutil import make_archive
make_archive('deploy-on-pynq-cnn-cifar10', 'zip', deployment_dir)

You can now download the created zipfile (**File -> Open**, mark the checkbox next to the `deploy-on-pynq-cnn-cifar10.zip` and select Download from the toolbar), then copy it to your PYNQ board (for instance via `scp` or `rsync`). Then, run the following commands **on the PYNQ board** to extract the archive and run the execution:

```shell
unzip deploy-on-pynq-cnn-cifar10.zip -d finn-cnn-cifar10-demo
cd finn-cnn-cifar10-demo
sudo python3 -m pip install bitstring
sudo python3 driver.py --exec_mode=execute --batchsize=1 --bitfile=resizer.bit --inputfile=input.npy
```

The output will be saved on the PYNQ board as `output.npy` and can be copied to the host and opened with `np.load()`.

### Validating the Accuracy on a PYNQ Board <a id='validation'></a>

**Ensure that your PYNQ board has a working internet connecting for the next steps, since there is some downloading involved.**

To validate the accuracy, we first need to install the [`dataset-loading`](https://github.com/fbcotter/dataset_loading) Python package to the PYNQ board. This will give us a convenient way of downloading and accessing the CIFAR-10 dataset.


Command to execute on PYNQ board:

```shell
sudo pip3 install git+https://github.com/fbcotter/dataset_loading.git@0.0.4#egg=dataset_loading
```

We can now use the `validate.py` script that was generated together with the driver to measure top-1 accuracy on the CIFAR-10 dataset.

Command to execute on PYNQ board:

```shell
sudo python3 validate.py --dataset cifar10 --batchsize 1000
```

### Throughput Test on PYNQ Board <a id='throughput'></a>
In addition to the functional verification, FINN also offers the possibility to measure the network performance directly on the PYNQ board. This can be done setting the `exec_mode` to `throughput_test`. 
Command to execute on PYNQ board:

```shell
sudo python3 driver.py --exec_mode=throughput_test --batchsize=1000 --bitfile=resizer.bit
```

The network metrics from the throughput test are saved in a file called `nw_metrics.txt` on the PYNQ board. Which can be investigated after running the command above.